# Module 2 — Colorectal Cancer HIPEC Agents: Exploration

This module profiles oxaliplatin and mitomycin C — the two primary agents used in hyperthermic
intraperitoneal chemotherapy (HIPEC) for peritoneal disease from colorectal and appendiceal primaries.
Unlike Module 1 (drug-first, systemic chemotherapy), HIPEC drugs are administered intraperitoneally
during surgery, which has important implications for FAERS reporting patterns and adverse event profiles.

**Analytical approach:** Drug-first. We identify oxaliplatin and mitomycin C reports in FAERS,
normalize drug name variants, filter to primary/secondary suspect roles, and build clean analysis
populations for each drug.

**Clinical context:** HIPEC combines cytoreductive surgery (CRS) with heated intraperitoneal
chemotherapy to treat peritoneal surface malignancies. Oxaliplatin-based HIPEC is common in
colorectal peritoneal metastases; mitomycin C is more widely used in appendiceal mucinous tumors
and mesothelioma. Adverse events from HIPEC are often distinct from systemic chemotherapy
given the route of administration and surgical context.

---

## Module Structure

| Notebook | Focus |
|---|---|
| `01_hipec_explore` | Drug name discovery, normalization, role code filtering, analysis population construction |
| `02_hipec_reactions` | Adverse reaction profiles for oxaliplatin and mitomycin C |
| `03_hipec_population` | Patient demographics - age, sex, country, indication profile |
| `04_hipec_outcomes` | Serious outcome distributions by drug |

**Prerequisite:** `faers.db` must be present at the path defined in the setup cell below.

In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\FDA_FAERS\database\faers.db"
conn = sqlite3.connect(db_path)

## Step 1: Drug Name Discovery

Scan the drug table for all oxaliplatin and mitomycin C name variants.
FAERS contains brand names, generic names, combination strings, and misspellings
that all need to be identified before building the analysis population.

In [2]:
# Scan for oxaliplatin name variants
oxaliplatin_variants = pd.read_sql_query("""
    SELECT drugname, COUNT(*) AS reports
    FROM drug
    WHERE drugname LIKE '%OXALIPLATIN%'
    GROUP BY drugname
    ORDER BY reports DESC
""", conn)

oxaliplatin_variants

,drugname,reports
0,OXALIPLATIN,13013
1,FLUOROURACIL\LEUCOVORIN CALCIUM\OXALIPLATIN,265
2,FLUOROURACIL\IRINOTECAN\LEUCOVORIN\OXALIPLATIN,55
3,CAPECITABINE\OXALIPLATIN,42
4,GEMCITABINE HYDROCHLORIDE\OXALIPLATIN,29
5,GEMCITABINE HYDROCHLORIDE\OXALIPLATIN\RITUXIMAB,21
6,FLUOROURACIL\LEUCOVORIN\OXALIPLATIN,12
7,OXALIPLATIN FOR INJECTION,10
8,OXALIPLATIN (ELOXATIN),8
9,SOX [GIMERACIL;OTERACIL;OXALIPLATIN;TEGAFUR],6


In [3]:
# Scan for mitomycin C name variants
mitomycin_variants = pd.read_sql_query("""
    SELECT drugname, COUNT(*) AS reports
    FROM drug
    WHERE drugname LIKE '%MITOMYCIN%'
    GROUP BY drugname
    ORDER BY reports DESC
""", conn)

mitomycin_variants

,drugname,reports
0,MITOMYCIN,198
1,MITOMYCIN C,55
2,MITOMYCIN-C,1
3,5 FU Mitomycine,1


## Step 2: Build Analysis Tables

Filter to PS/SS role codes (primary and secondary suspect drugs only) and store
as `oxaliplatin_analysis` and `mitomycin_analysis` tables. These are the anchor
tables used across all subsequent notebooks in this module.

Review the variant lists above and add any additional drugname patterns needed
to the LIKE filters below before running.

In [4]:
# Build oxaliplatin analysis table
conn.execute("DROP TABLE IF EXISTS oxaliplatin_analysis;")
conn.execute("""
    CREATE TABLE oxaliplatin_analysis AS
    SELECT *
    FROM drug
    WHERE drugname LIKE '%OXALIPLATIN%'
      AND role_cod IN ('PS', 'SS')
""")
conn.commit()

total_oxa = conn.execute("SELECT COUNT(DISTINCT primaryid) FROM oxaliplatin_analysis").fetchone()[0]
print(f"Oxaliplatin analysis reports: {total_oxa:,}")

Oxaliplatin analysis reports: 7,310


In [5]:
# Build mitomycin analysis table
conn.execute("DROP TABLE IF EXISTS mitomycin_analysis;")
conn.execute("""
    CREATE TABLE mitomycin_analysis AS
    SELECT *
    FROM drug
    WHERE drugname LIKE '%MITOMYCIN%'
      AND role_cod IN ('PS', 'SS')
""")
conn.commit()

total_mmc = conn.execute("SELECT COUNT(DISTINCT primaryid) FROM mitomycin_analysis").fetchone()[0]
print(f"Mitomycin analysis reports: {total_mmc:,}")

Mitomycin analysis reports: 133


## Step 3: Role Code Distribution

Verify the PS/SS breakdown for each drug. A high proportion of PS (primary suspect)
reports indicates strong causality attribution by the reporter.

In [6]:
# Role code breakdown for oxaliplatin
oxa_roles = pd.read_sql_query("""
    SELECT role_cod, COUNT(*) AS reports
    FROM drug
    WHERE drugname LIKE '%OXALIPLATIN%'
    GROUP BY role_cod
    ORDER BY reports DESC
""", conn)

print("Oxaliplatin role codes:")
print(oxa_roles)

# Role code breakdown for mitomycin
mmc_roles = pd.read_sql_query("""
    SELECT role_cod, COUNT(*) AS reports
    FROM drug
    WHERE drugname LIKE '%MITOMYCIN%'
    GROUP BY role_cod
    ORDER BY reports DESC
""", conn)

print("\nMitomycin role codes:")
print(mmc_roles)

Oxaliplatin role codes:
  role_cod  reports
0       SS     7313
1       PS     4252
2        C     1867
3        I       45

Mitomycin role codes:
  role_cod  reports
0       SS      133
1        C       74
2       PS       48


## Step 4: Indication Profile

What indications are these drugs being reported for? Oxaliplatin is used broadly
in colorectal cancer beyond HIPEC, so indication filtering helps identify
the peritoneal disease subgroup specifically.

In [7]:
# Top indications for oxaliplatin
oxa_indications = pd.read_sql_query("""
    SELECT i.indi_pt, COUNT(DISTINCT i.primaryid) AS reports
    FROM indi i
    JOIN oxaliplatin_analysis o ON i.primaryid = o.primaryid
    GROUP BY i.indi_pt
    ORDER BY reports DESC
    LIMIT 20
""", conn)

print("Top oxaliplatin indications:")
print(oxa_indications)

# Top indications for mitomycin
mmc_indications = pd.read_sql_query("""
    SELECT i.indi_pt, COUNT(DISTINCT i.primaryid) AS reports
    FROM indi i
    JOIN mitomycin_analysis m ON i.primaryid = m.primaryid
    GROUP BY i.indi_pt
    ORDER BY reports DESC
    LIMIT 20
""", conn)

print("\nTop mitomycin indications:")
print(mmc_indications)

Top oxaliplatin indications:
                                indi_pt  reports
0   Product used for unknown indication     1485
1                          Colon cancer      655
2                        Gastric cancer      557
3          Colorectal cancer metastatic      490
4         Diffuse large B-cell lymphoma      416
5               Adenocarcinoma of colon      394
6                Adenocarcinoma gastric      366
7            Oesophageal adenocarcinoma      355
8                     Colorectal cancer      308
9                          Chemotherapy      292
10                        Rectal cancer      280
11                          Prophylaxis      227
12                Rectal adenocarcinoma      220
13                  Metastases to liver      199
14                 Pancreatic carcinoma      183
15                        Premedication      153
16              Adenocarcinoma pancreas      146
17                         Hypertension      123
18              Colon cancer metastatic 